# JDGenRec: GenRec Paper Implementation
**GenRec: A Preference-Oriented Generative Framework for Large-Scale Recommendation**
(Zou et al., SIGIR 2026) — [arXiv:2604.14878](https://arxiv.org/abs/2604.14878)

This notebook runs the full pipeline on a free Colab T4 GPU:
1. Setup & install dependencies
2. Train RQVAE (Semantic ID generation)
3. Train GenRec with Page-wise NTP + Token Merger
4. Evaluate with constrained beam search

**Requirements**: GPU runtime (T4 is sufficient). Go to Runtime → Change runtime type → T4 GPU

## 1. Setup

In [ ]:
# Verify GPU is available
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    cap = torch.cuda.get_device_capability()
    print(f"Compute capability: {cap[0]}.{cap[1]}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    dtype = torch.bfloat16 if cap[0] >= 8 else torch.float16
    print(f"Will use dtype: {dtype}")

In [ ]:
# Install dependencies
!pip install -q transformers>=4.40.0 gin-config wandb tqdm accelerate scipy

In [ ]:
# Create project structure
import os
for d in ['src/models', 'src/data', 'src/modules', 'src/trainers', 'checkpoints', 'data']:
    os.makedirs(f'/content/JDGenRec/{d}', exist_ok=True)
os.chdir('/content/JDGenRec')
print(f"Working directory: {os.getcwd()}")

## 2. Create Source Files

In [ ]:
%%writefile src/__init__.py
"""JDGenRec"""
__version__ = "0.1.0"

In [ ]:
%%writefile src/modules/__init__.py
from src.modules.token_merger import TokenMerger
from src.modules.metrics import TopKAccumulator, compute_recall_at_k, compute_ndcg_at_k
from src.modules.beam_search import constrained_beam_search

In [ ]:
%%writefile src/modules/token_merger.py
"""Token Merger: compresses multi-token Semantic IDs into a single latent vector."""
import torch
from torch import nn


class TokenMerger(nn.Module):
    """
    Linear Token Merger from the GenRec paper.
    h_vi = Linear(Concat(e(si^1), e(si^2), e(si^3)))
    """
    def __init__(self, embed_dim: int, num_codes: int = 3):
        super().__init__()
        self.num_codes = num_codes
        self.proj = nn.Linear(embed_dim * num_codes, embed_dim, bias=False)

    def forward(self, sid_embeddings: torch.Tensor) -> torch.Tensor:
        """
        Args:
            sid_embeddings: [batch, num_items, num_codes, embed_dim]
        Returns:
            merged: [batch, num_items, embed_dim]
        """
        batch, num_items, num_codes, embed_dim = sid_embeddings.shape
        concatenated = sid_embeddings.view(batch, num_items, num_codes * embed_dim)
        merged = self.proj(concatenated)
        return merged

In [ ]:
%%writefile src/modules/metrics.py
"""Evaluation metrics for recommendation: Recall@K and NDCG@K."""
import math
from typing import List, Dict


def compute_recall_at_k(predictions: List[int], ground_truth: int, k: int) -> float:
    return 1.0 if ground_truth in predictions[:k] else 0.0


def compute_ndcg_at_k(predictions: List[int], ground_truth: int, k: int) -> float:
    for i, pred in enumerate(predictions[:k]):
        if pred == ground_truth:
            return 1.0 / math.log2(i + 2)
    return 0.0


class TopKAccumulator:
    def __init__(self, ks: List[int] = [5, 10]):
        self.ks = ks
        self.reset()

    def reset(self):
        self.recall = {k: 0.0 for k in self.ks}
        self.ndcg = {k: 0.0 for k in self.ks}
        self.count = 0

    def update(self, predictions: List[int], ground_truth: int):
        self.count += 1
        for k in self.ks:
            self.recall[k] += compute_recall_at_k(predictions, ground_truth, k)
            self.ndcg[k] += compute_ndcg_at_k(predictions, ground_truth, k)

    def compute(self) -> Dict[str, float]:
        if self.count == 0:
            return {}
        results = {}
        for k in self.ks:
            results[f"Recall@{k}"] = self.recall[k] / self.count
            results[f"NDCG@{k}"] = self.ndcg[k] / self.count
        return results

In [ ]:
%%writefile src/modules/beam_search.py
"""Constrained beam search for generating Semantic ID sequences."""
import torch
from torch.nn import functional as F
from typing import Tuple
from transformers import DynamicCache


def constrained_beam_search(
    model, input_ids, attention_mask, position_mask, code_map,
    num_codebooks, beam_width=20, topk=10,
) -> Tuple[torch.Tensor, torch.Tensor]:
    B, L = input_ids.shape
    device = input_ids.device
    num_steps = num_codebooks + 1
    position_mask = position_mask.to(device)
    code_map = code_map.to(device)

    out = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=True)
    logits = out.logits[:, -1, :]
    past_kv = out.past_key_values

    mask_0 = position_mask[0].unsqueeze(0)
    logits = logits.masked_fill(~mask_0, float('-inf'))
    log_probs = F.log_softmax(logits, dim=-1)
    topk_scores, topk_ids = log_probs.topk(beam_width, dim=-1)
    beam_scores = topk_scores
    beam_tokens = topk_ids.unsqueeze(-1)

    past_kv = _expand_past_kv(past_kv, beam_width)
    attn_mask = attention_mask.unsqueeze(1).expand(-1, beam_width, -1).reshape(B * beam_width, L)

    for step in range(1, num_steps):
        next_input = beam_tokens[:, :, -1].reshape(B * beam_width, 1)
        attn_mask = torch.cat([attn_mask, torch.ones(B * beam_width, 1, device=device, dtype=attn_mask.dtype)], dim=-1)
        out = model(input_ids=next_input, attention_mask=attn_mask, past_key_values=past_kv, use_cache=True)
        logits = out.logits[:, -1, :]
        past_kv = out.past_key_values
        mask_s = position_mask[step].unsqueeze(0)
        logits = logits.masked_fill(~mask_s, float('-inf'))
        log_probs = F.log_softmax(logits, dim=-1).view(B, beam_width, -1)

        if step < num_codebooks:
            candidate_scores = beam_scores.unsqueeze(-1) + log_probs
            candidate_scores = candidate_scores.view(B, -1)
            top_scores, top_indices = candidate_scores.topk(beam_width, dim=-1)
            beam_idx = top_indices // log_probs.size(-1)
            token_idx = top_indices % log_probs.size(-1)
            beam_scores = top_scores
            prev_tokens = torch.gather(beam_tokens, 1, beam_idx.unsqueeze(-1).expand(-1, -1, beam_tokens.size(-1)))
            beam_tokens = torch.cat([prev_tokens, token_idx.unsqueeze(-1)], dim=-1)
            reorder_idx = (torch.arange(B, device=device).unsqueeze(1) * beam_width + beam_idx).view(-1)
            past_kv = _reorder_past_kv(past_kv, reorder_idx)
            attn_mask = attn_mask[reorder_idx]
        else:
            allowed_mask = position_mask[step]
            eos_token_idx = allowed_mask.nonzero(as_tuple=False)[0, 0]
            eos_log_prob = log_probs[:, :, eos_token_idx]
            beam_scores = beam_scores + eos_log_prob

    sem_ids = torch.zeros(B, beam_width, num_codebooks, dtype=torch.long, device=device)
    for c in range(num_codebooks):
        token_ids = beam_tokens[:, :, c]
        sem_ids[:, :, c] = code_map[c][token_ids]

    sorted_idx = beam_scores.argsort(dim=-1, descending=True)[:, :topk]
    sem_ids = torch.gather(sem_ids, 1, sorted_idx.unsqueeze(-1).expand(-1, -1, num_codebooks))
    scores = torch.gather(beam_scores, 1, sorted_idx)
    return sem_ids, scores


def _expand_past_kv(past_kv, beam_width):
    if isinstance(past_kv, DynamicCache):
        new_cache = DynamicCache()
        for layer_idx in range(len(past_kv)):
            key = past_kv.key_cache[layer_idx]
            value = past_kv.value_cache[layer_idx]
            new_key = key.unsqueeze(1).expand(-1, beam_width, -1, -1, -1).reshape(-1, *key.shape[1:])
            new_value = value.unsqueeze(1).expand(-1, beam_width, -1, -1, -1).reshape(-1, *value.shape[1:])
            new_cache.update(new_key, new_value, layer_idx)
        return new_cache
    return tuple(
        tuple(t.unsqueeze(1).expand(-1, beam_width, -1, -1, -1).reshape(t.size(0) * beam_width, *t.shape[1:]) for t in layer_kv)
        for layer_kv in past_kv
    )


def _reorder_past_kv(past_kv, reorder_idx):
    if isinstance(past_kv, DynamicCache):
        past_kv.reorder_cache(reorder_idx)
        return past_kv
    return tuple(
        tuple(t.index_select(0, reorder_idx) for t in layer_kv)
        for layer_kv in past_kv
    )

In [ ]:
%%writefile src/models/__init__.py
from src.models.rqvae import RQVAE
from src.models.genrec import GenRec

In [ ]:
%%writefile src/models/rqvae.py
"""Residual Quantized VAE for Semantic ID generation."""
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Optional


class VectorQuantizer(nn.Module):
    def __init__(self, codebook_size, embed_dim, commitment_cost=0.25, decay=0.99, eps=1e-5):
        super().__init__()
        self.codebook_size = codebook_size
        self.embed_dim = embed_dim
        self.commitment_cost = commitment_cost
        self.decay = decay
        self.eps = eps
        self.embedding = nn.Embedding(codebook_size, embed_dim)
        self.embedding.weight.data.uniform_(-1.0 / codebook_size, 1.0 / codebook_size)
        self.register_buffer("cluster_size", torch.zeros(codebook_size))
        self.register_buffer("ema_embed", self.embedding.weight.data.clone())

    def forward(self, z):
        distances = (z.pow(2).sum(-1, keepdim=True) + self.embedding.weight.pow(2).sum(-1)
                     - 2 * z @ self.embedding.weight.t())
        indices = distances.argmin(dim=-1)
        quantized = self.embedding(indices)
        if self.training:
            encodings = F.one_hot(indices, self.codebook_size).float()
            self.cluster_size.data.mul_(self.decay).add_(encodings.sum(0), alpha=1 - self.decay)
            embed_sum = encodings.t() @ z
            self.ema_embed.data.mul_(self.decay).add_(embed_sum, alpha=1 - self.decay)
            n = self.cluster_size.sum()
            cluster_size = (self.cluster_size + self.eps) / (n + self.codebook_size * self.eps) * n
            self.embedding.weight.data.copy_(self.ema_embed / cluster_size.unsqueeze(-1))
            # Dead code revival
            dead_mask = self.cluster_size < 1.0
            num_dead = dead_mask.sum().item()
            if num_dead > 0 and z.size(0) > 0:
                num_replace = min(num_dead, z.size(0))
                rand_idx = torch.randperm(z.size(0), device=z.device)[:num_replace]
                dead_idx = dead_mask.nonzero(as_tuple=True)[0][:num_replace]
                self.embedding.weight.data[dead_idx] = z[rand_idx].detach()
                self.ema_embed.data[dead_idx] = z[rand_idx].detach()
                self.cluster_size.data[dead_idx] = 1.0
        commitment_loss = self.commitment_cost * F.mse_loss(z, quantized.detach())
        quantized = z + (quantized - z).detach()
        return quantized, indices, commitment_loss


class RQVAE(nn.Module):
    def __init__(self, input_dim, embedding_dim=256, num_codebooks=3, codebook_size=256,
                 hidden_dims=None, commitment_cost=0.25):
        super().__init__()
        self.input_dim = input_dim
        self.embedding_dim = embedding_dim
        self.num_codebooks = num_codebooks
        self.codebook_size = codebook_size
        if hidden_dims is None:
            hidden_dims = [512, 256]
        encoder_layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            encoder_layers.extend([nn.Linear(prev_dim, h_dim), nn.LayerNorm(h_dim), nn.GELU()])
            prev_dim = h_dim
        encoder_layers.append(nn.Linear(prev_dim, embedding_dim))
        self.encoder = nn.Sequential(*encoder_layers)
        decoder_layers = []
        prev_dim = embedding_dim
        for h_dim in reversed(hidden_dims):
            decoder_layers.extend([nn.Linear(prev_dim, h_dim), nn.LayerNorm(h_dim), nn.GELU()])
            prev_dim = h_dim
        decoder_layers.append(nn.Linear(prev_dim, input_dim))
        self.decoder = nn.Sequential(*decoder_layers)
        self.quantizers = nn.ModuleList([
            VectorQuantizer(codebook_size, embedding_dim, commitment_cost)
            for _ in range(num_codebooks)
        ])

    def encode(self, x): return self.encoder(x)
    def decode(self, z): return self.decoder(z)

    def quantize(self, z):
        residual = z
        quantized_sum = torch.zeros_like(z)
        all_indices = []
        total_loss = torch.tensor(0.0, device=z.device)
        for quantizer in self.quantizers:
            quantized, indices, loss = quantizer(residual)
            quantized_sum = quantized_sum + quantized
            residual = residual - quantized.detach()
            all_indices.append(indices)
            total_loss = total_loss + loss
        return quantized_sum, torch.stack(all_indices, dim=-1), total_loss

    def forward(self, x):
        z = self.encode(x)
        quantized, indices, commit_loss = self.quantize(z)
        recon = self.decode(quantized)
        recon_loss = F.mse_loss(recon, x)
        return recon, indices, commit_loss, recon_loss

    @torch.no_grad()
    def get_codes(self, x):
        z = self.encode(x)
        _, indices, _ = self.quantize(z)
        return indices

    def init_codebook_with_kmeans(self, data, n_iter=20):
        with torch.no_grad():
            z = self.encode(data)
            residual = z
            for quantizer in self.quantizers:
                perm = torch.randperm(z.size(0))[:quantizer.codebook_size]
                centroids = residual[perm].clone()
                for _ in range(n_iter):
                    dists = (residual.pow(2).sum(-1, keepdim=True) + centroids.pow(2).sum(-1)
                             - 2 * residual @ centroids.t())
                    assignments = dists.argmin(dim=-1)
                    for k in range(quantizer.codebook_size):
                        mask = assignments == k
                        if mask.sum() > 0:
                            centroids[k] = residual[mask].mean(dim=0)
                quantizer.embedding.weight.data.copy_(centroids)
                quantizer.ema_embed.data.copy_(centroids)
                quantized = quantizer.embedding(assignments)
                residual = residual - quantized

In [ ]:
%%writefile src/models/genrec.py
"""GenRec: Decoder-only architecture with Token Merger and Page-wise NTP."""
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, Optional, Tuple
from transformers import AutoTokenizer, AutoModelForCausalLM, DynamicCache
from src.modules.token_merger import TokenMerger


class GenRec(nn.Module):
    def __init__(self, pretrained_path, num_codebooks=3, codebook_size=256, use_token_merger=True):
        super().__init__()
        self.num_codebooks = num_codebooks
        self.codebook_size = codebook_size
        self.use_token_merger = use_token_merger

        self.tokenizer = AutoTokenizer.from_pretrained(pretrained_path, trust_remote_code=True)
        # bf16 on Ampere+ (capability >= 8), fp16 on T4/V100, float32 on CPU
        if torch.cuda.is_available():
            cap = torch.cuda.get_device_capability()
            dtype = torch.bfloat16 if cap[0] >= 8 else torch.float16
        else:
            dtype = torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(pretrained_path, trust_remote_code=True, torch_dtype=dtype)

        # Add codebook tokens
        for c in range(num_codebooks):
            for k in range(codebook_size):
                self.tokenizer.add_special_tokens({"additional_special_tokens": [f"<C{c}_{k}>"]})
        self.model.resize_token_embeddings(len(self.tokenizer))

        # Structural tokens
        self.tokenizer.add_special_tokens({"additional_special_tokens": ["<sep>", "<page>", "<bos_rec>", "<eos_rec>"]})
        self.model.resize_token_embeddings(len(self.tokenizer))

        self.sep_token_id = self.tokenizer.convert_tokens_to_ids("<sep>")
        self.page_token_id = self.tokenizer.convert_tokens_to_ids("<page>")
        self.bos_rec_id = self.tokenizer.convert_tokens_to_ids("<bos_rec>")
        self.eos_rec_id = self.tokenizer.convert_tokens_to_ids("<eos_rec>")

        self.codebook_token_ids = []
        for c in range(num_codebooks):
            level_ids = [self.tokenizer.convert_tokens_to_ids(f"<C{c}_{k}>") for k in range(codebook_size)]
            self.codebook_token_ids.append(level_ids)

        hidden_size = self.model.config.hidden_size
        if use_token_merger:
            self.token_merger = TokenMerger(embed_dim=hidden_size, num_codes=num_codebooks).to(dtype)

        self._build_position_mask()

    def _build_position_mask(self):
        vocab_size = len(self.tokenizer)
        num_steps = self.num_codebooks + 1
        position_mask = torch.zeros(num_steps, vocab_size, dtype=torch.bool)
        for step in range(self.num_codebooks):
            for token_id in self.codebook_token_ids[step]:
                position_mask[step, token_id] = True
        position_mask[self.num_codebooks, self.sep_token_id] = True
        position_mask[self.num_codebooks, self.eos_rec_id] = True
        if self.tokenizer.eos_token_id is not None:
            position_mask[self.num_codebooks, self.tokenizer.eos_token_id] = True
        self.register_buffer("position_mask", position_mask)
        code_map = torch.zeros(self.num_codebooks, vocab_size, dtype=torch.long)
        for c in range(self.num_codebooks):
            for k in range(self.codebook_size):
                code_map[c, self.codebook_token_ids[c][k]] = k
        self.register_buffer("code_map", code_map)

    def gradient_checkpointing_enable(self):
        self.model.gradient_checkpointing_enable()

    def get_input_embeddings(self):
        return self.model.get_input_embeddings()

    def codes_to_token_ids(self, codes):
        token_ids = torch.zeros_like(codes)
        for c in range(self.num_codebooks):
            level_ids = torch.tensor(self.codebook_token_ids[c], device=codes.device)
            token_ids[..., c] = level_ids[codes[..., c]]
        return token_ids

    def build_sft_inputs(self, input_codes, target_codes, input_lengths, target_lengths):
        B = input_codes.shape[0]
        device = input_codes.device
        input_token_ids = self.codes_to_token_ids(input_codes)
        target_token_ids = self.codes_to_token_ids(target_codes)
        all_input_ids, all_labels = [], []

        for b in range(B):
            hist_len = input_lengths[b].item()
            tgt_len = target_lengths[b].item()
            if self.use_token_merger:
                prompt_ids = [self.bos_rec_id]
                for i in range(hist_len):
                    prompt_ids.append(input_token_ids[b, i, 0].item())
                    if i < hist_len - 1:
                        prompt_ids.append(self.sep_token_id)
                prompt_ids.append(self.page_token_id)
            else:
                prompt_ids = [self.bos_rec_id]
                for i in range(hist_len):
                    for c in range(self.num_codebooks):
                        prompt_ids.append(input_token_ids[b, i, c].item())
                    if i < hist_len - 1:
                        prompt_ids.append(self.sep_token_id)
                prompt_ids.append(self.page_token_id)

            target_ids = []
            for i in range(tgt_len):
                for c in range(self.num_codebooks):
                    target_ids.append(target_token_ids[b, i, c].item())
                if i < tgt_len - 1:
                    target_ids.append(self.sep_token_id)
            target_ids.append(self.eos_rec_id)

            full_ids = prompt_ids + target_ids
            labels = [-100] * len(prompt_ids) + target_ids
            all_input_ids.append(torch.tensor(full_ids, dtype=torch.long, device=device))
            all_labels.append(torch.tensor(labels, dtype=torch.long, device=device))

        max_len = max(ids.size(0) for ids in all_input_ids)
        pad_id = self.tokenizer.pad_token_id or 0
        input_ids = torch.full((B, max_len), pad_id, dtype=torch.long, device=device)
        attention_mask = torch.zeros(B, max_len, dtype=torch.long, device=device)
        labels = torch.full((B, max_len), -100, dtype=torch.long, device=device)
        for b in range(B):
            sl = all_input_ids[b].size(0)
            input_ids[b, :sl] = all_input_ids[b]
            attention_mask[b, :sl] = 1
            labels[b, :sl] = all_labels[b]
        return input_ids, attention_mask, labels

    def forward_sft(self, input_codes, target_codes, input_lengths, target_lengths):
        input_ids, attention_mask, labels = self.build_sft_inputs(
            input_codes, target_codes, input_lengths, target_lengths
        )
        if self.use_token_merger:
            inputs_embeds = self.get_input_embeddings()(input_ids)
            B = input_codes.shape[0]
            for b in range(B):
                hist_len = input_lengths[b].item()
                input_token_ids_b = self.codes_to_token_ids(input_codes[b, :hist_len].unsqueeze(0))
                sid_embeds = self.get_input_embeddings()(input_token_ids_b.squeeze(0))
                merged = self.token_merger(sid_embeds.unsqueeze(0)).squeeze(0)
                pos = 1
                for i in range(hist_len):
                    inputs_embeds[b, pos] = merged[i]
                    pos += 1
                    if i < hist_len - 1:
                        pos += 1
            outputs = self.model(inputs_embeds=inputs_embeds, attention_mask=attention_mask, labels=labels, use_cache=False)
        else:
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask, labels=labels, use_cache=False)
        return {"loss": outputs.loss, "logits": outputs.logits}

    def build_eval_inputs(self, input_codes, input_lengths):
        B = input_codes.shape[0]
        device = input_codes.device
        input_token_ids = self.codes_to_token_ids(input_codes)
        all_input_ids = []
        for b in range(B):
            hist_len = input_lengths[b].item()
            if self.use_token_merger:
                ids = [self.bos_rec_id]
                for i in range(hist_len):
                    ids.append(input_token_ids[b, i, 0].item())
                    if i < hist_len - 1:
                        ids.append(self.sep_token_id)
                ids.append(self.page_token_id)
            else:
                ids = [self.bos_rec_id]
                for i in range(hist_len):
                    for c in range(self.num_codebooks):
                        ids.append(input_token_ids[b, i, c].item())
                    if i < hist_len - 1:
                        ids.append(self.sep_token_id)
                ids.append(self.page_token_id)
            all_input_ids.append(torch.tensor(ids, dtype=torch.long, device=device))

        max_len = max(ids.size(0) for ids in all_input_ids)
        pad_id = self.tokenizer.pad_token_id or 0
        input_ids = torch.full((B, max_len), pad_id, dtype=torch.long, device=device)
        attention_mask = torch.zeros(B, max_len, dtype=torch.long, device=device)
        for b in range(B):
            sl = all_input_ids[b].size(0)
            input_ids[b, :sl] = all_input_ids[b]
            attention_mask[b, :sl] = 1
        return input_ids, attention_mask

    @torch.no_grad()
    def generate_beam(self, input_codes, input_lengths, beam_width=20, topk=10):
        from src.modules.beam_search import constrained_beam_search, _expand_past_kv, _reorder_past_kv
        input_ids, attention_mask = self.build_eval_inputs(input_codes, input_lengths)

        if self.use_token_merger:
            inputs_embeds = self.get_input_embeddings()(input_ids)
            B = input_codes.shape[0]
            for b in range(B):
                hist_len = input_lengths[b].item()
                input_token_ids_b = self.codes_to_token_ids(input_codes[b, :hist_len].unsqueeze(0))
                sid_embeds = self.get_input_embeddings()(input_token_ids_b.squeeze(0))
                merged = self.token_merger(sid_embeds.unsqueeze(0)).squeeze(0)
                pos = 1
                for i in range(hist_len):
                    inputs_embeds[b, pos] = merged[i]
                    pos += 1
                    if i < hist_len - 1:
                        pos += 1

            out = self.model(inputs_embeds=inputs_embeds, attention_mask=attention_mask, use_cache=True)
            logits = out.logits[:, -1, :]
            past_kv = out.past_key_values
            return self._beam_from_logits(logits, past_kv, attention_mask, beam_width, topk, B)
        else:
            return constrained_beam_search(
                self.model, input_ids, attention_mask,
                self.position_mask, self.code_map,
                self.num_codebooks, beam_width, topk
            )

    def _beam_from_logits(self, first_logits, past_kv, attention_mask, beam_width, topk, B):
        from src.modules.beam_search import _expand_past_kv, _reorder_past_kv
        L = attention_mask.shape[1]
        device = first_logits.device

        mask_0 = self.position_mask[0].unsqueeze(0)
        first_logits = first_logits.masked_fill(~mask_0, float('-inf'))
        log_probs = F.log_softmax(first_logits, dim=-1)
        topk_scores, topk_ids = log_probs.topk(beam_width, dim=-1)
        beam_scores = topk_scores
        beam_tokens = topk_ids.unsqueeze(-1)

        past_kv = _expand_past_kv(past_kv, beam_width)
        attn_mask = attention_mask.unsqueeze(1).expand(-1, beam_width, -1).reshape(B * beam_width, L)

        for step in range(1, self.num_codebooks + 1):
            next_input = beam_tokens[:, :, -1].reshape(B * beam_width, 1)
            attn_mask = torch.cat([attn_mask, torch.ones(B * beam_width, 1, device=device, dtype=attn_mask.dtype)], dim=-1)
            out = self.model(input_ids=next_input, attention_mask=attn_mask, past_key_values=past_kv, use_cache=True)
            logits = out.logits[:, -1, :]
            past_kv = out.past_key_values
            mask_s = self.position_mask[step].unsqueeze(0)
            logits = logits.masked_fill(~mask_s, float('-inf'))
            log_probs = F.log_softmax(logits, dim=-1).view(B, beam_width, -1)

            if step < self.num_codebooks:
                candidate_scores = beam_scores.unsqueeze(-1) + log_probs
                candidate_scores = candidate_scores.view(B, -1)
                top_scores, top_indices = candidate_scores.topk(beam_width, dim=-1)
                beam_idx = top_indices // log_probs.size(-1)
                token_idx = top_indices % log_probs.size(-1)
                beam_scores = top_scores
                prev_tokens = torch.gather(beam_tokens, 1, beam_idx.unsqueeze(-1).expand(-1, -1, beam_tokens.size(-1)))
                beam_tokens = torch.cat([prev_tokens, token_idx.unsqueeze(-1)], dim=-1)
                reorder_idx = (torch.arange(B, device=device).unsqueeze(1) * beam_width + beam_idx).view(-1)
                past_kv = _reorder_past_kv(past_kv, reorder_idx)
                attn_mask = attn_mask[reorder_idx]
            else:
                allowed_mask = self.position_mask[step]
                eos_token_idx = allowed_mask.nonzero(as_tuple=False)[0, 0]
                beam_scores = beam_scores + log_probs[:, :, eos_token_idx]

        sem_ids = torch.zeros(B, beam_width, self.num_codebooks, dtype=torch.long, device=device)
        for c in range(self.num_codebooks):
            sem_ids[:, :, c] = self.code_map[c][beam_tokens[:, :, c]]
        sorted_idx = beam_scores.argsort(dim=-1, descending=True)[:, :topk]
        sem_ids = torch.gather(sem_ids, 1, sorted_idx.unsqueeze(-1).expand(-1, -1, self.num_codebooks))
        scores = torch.gather(beam_scores, 1, sorted_idx)
        return sem_ids, scores

    def save_pretrained(self, save_dir, **kwargs):
        import os
        os.makedirs(save_dir, exist_ok=True)
        self.model.save_pretrained(save_dir, **kwargs)
        self.tokenizer.save_pretrained(save_dir)
        if self.use_token_merger:
            torch.save(self.token_merger.state_dict(), os.path.join(save_dir, "token_merger.pt"))

In [ ]:
%%writefile src/data/__init__.py
from src.data.amazon import AmazonDataset
from src.data.genrec_dataset import GenRecSFTDataset, GenRecEvalDataset

In [ ]:
%%writefile src/data/amazon.py
"""Amazon 2014 dataset loading and preprocessing."""
import os, json, gzip, pickle
import numpy as np
from collections import defaultdict
from typing import Dict, List, Tuple
from urllib.request import urlretrieve

AMAZON_URLS = {
    "beauty": "http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Beauty_5.json.gz",
    "sports": "http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Sports_and_Outdoors_5.json.gz",
    "toys": "http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Toys_and_Games_5.json.gz",
}

class AmazonDataset:
    def __init__(self, data_dir="./data", split="beauty", max_seq_len=50):
        self.data_dir = data_dir
        self.split = split
        self.max_seq_len = max_seq_len
        self.user_sequences = {}
        self.num_users = 0
        self.num_items = 0
        self.item_id_map = {}
        self.user_id_map = {}
        self._load_or_download()

    def _load_or_download(self):
        cache_path = os.path.join(self.data_dir, f"amazon_{self.split}_processed.pkl")
        if os.path.exists(cache_path):
            with open(cache_path, "rb") as f:
                data = pickle.load(f)
            self.user_sequences = data["user_sequences"]
            self.num_users = data["num_users"]
            self.num_items = data["num_items"]
            self.item_id_map = data["item_id_map"]
            self.user_id_map = data["user_id_map"]
            return
        os.makedirs(self.data_dir, exist_ok=True)
        raw_path = os.path.join(self.data_dir, f"reviews_{self.split}_5.json.gz")
        if not os.path.exists(raw_path):
            url = AMAZON_URLS[self.split]
            print(f"Downloading {self.split} dataset...")
            urlretrieve(url, raw_path)
        self._preprocess(raw_path, cache_path)

    def _preprocess(self, raw_path, cache_path):
        interactions = []
        with gzip.open(raw_path, "rt", encoding="utf-8") as f:
            for line in f:
                r = json.loads(line)
                interactions.append((r["reviewerID"], r["asin"], r["unixReviewTime"]))
        interactions.sort(key=lambda x: x[2])
        user_items = defaultdict(list)
        for user, item, time in interactions:
            user_items[user].append((item, time))
        for _ in range(10):
            item_count = defaultdict(int)
            for items in user_items.values():
                for item, _ in items:
                    item_count[item] += 1
            valid_items = {i for i, c in item_count.items() if c >= 5}
            new_user_items = {}
            for user, items in user_items.items():
                filtered = [(i, t) for i, t in items if i in valid_items]
                if len(filtered) >= 5:
                    new_user_items[user] = filtered
            user_items = new_user_items
        all_items = set()
        for items in user_items.values():
            for item, _ in items:
                all_items.add(item)
        self.item_id_map = {item: idx + 1 for idx, item in enumerate(sorted(all_items))}
        self.user_id_map = {user: idx for idx, user in enumerate(sorted(user_items.keys()))}
        self.user_sequences = {}
        for user, items in user_items.items():
            items_sorted = sorted(items, key=lambda x: x[1])
            self.user_sequences[self.user_id_map[user]] = [self.item_id_map[i] for i, _ in items_sorted]
        self.num_users = len(self.user_sequences)
        self.num_items = len(self.item_id_map)
        with open(cache_path, "wb") as f:
            pickle.dump({"user_sequences": self.user_sequences, "num_users": self.num_users,
                         "num_items": self.num_items, "item_id_map": self.item_id_map,
                         "user_id_map": self.user_id_map}, f)
        print(f"Preprocessed {self.split}: {self.num_users} users, {self.num_items} items")

    def get_splits(self):
        train_seqs, val_targets, test_targets = {}, {}, {}
        for uid, seq in self.user_sequences.items():
            if len(seq) < 3: continue
            train_seqs[uid] = seq[:-2]
            val_targets[uid] = seq[-2]
            test_targets[uid] = seq[-1]
        return train_seqs, val_targets, test_targets

    def get_item_embeddings(self, embedding_dim=64):
        from scipy.sparse import csr_matrix
        from scipy.sparse.linalg import svds
        rows, cols = [], []
        for uid, items in self.user_sequences.items():
            for iid in items:
                rows.append(uid - 1)
                cols.append(iid - 1)
        data = np.ones(len(rows), dtype=np.float32)
        mat = csr_matrix((data, (rows, cols)), shape=(self.num_users, self.num_items))
        k = min(embedding_dim, min(self.num_users, self.num_items) - 1)
        _, s, Vt = svds(mat, k=k)
        item_vecs = (Vt.T * np.sqrt(s)).astype(np.float32)
        if k < embedding_dim:
            pad = np.zeros((self.num_items, embedding_dim - k), dtype=np.float32)
            item_vecs = np.concatenate([item_vecs, pad], axis=1)
        norms = np.linalg.norm(item_vecs, axis=1, keepdims=True)
        item_vecs = item_vecs / np.maximum(norms, 1e-8)
        emb = np.zeros((self.num_items + 1, embedding_dim), dtype=np.float32)
        emb[1:] = item_vecs
        return emb

In [ ]:
%%writefile src/data/genrec_dataset.py
"""GenRec Page-wise SFT Dataset and Evaluation Dataset."""
import torch
from torch.utils.data import Dataset
from typing import Dict, List
import numpy as np


class GenRecSFTDataset(Dataset):
    def __init__(self, user_sequences, item_codes, num_codebooks=3, codebook_size=256,
                 max_seq_len=50, page_size=3):
        self.num_codebooks = num_codebooks
        self.item_codes = item_codes
        self.samples = []
        for user_id, seq in user_sequences.items():
            if len(seq) < page_size + 1: continue
            for end_idx in range(page_size, len(seq)):
                start_idx = max(0, end_idx - page_size - max_seq_len)
                history = seq[start_idx:end_idx - page_size]
                page_items = seq[end_idx - page_size:end_idx]
                if len(history) > 0:
                    self.samples.append((history[-max_seq_len:], page_items))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        history, page_items = self.samples[idx]
        input_codes = np.array([self.item_codes[i] for i in history])
        target_codes = np.array([self.item_codes[i] for i in page_items])
        return {"input_codes": torch.LongTensor(input_codes), "target_codes": torch.LongTensor(target_codes),
                "input_length": len(history), "target_length": len(page_items)}


class GenRecEvalDataset(Dataset):
    def __init__(self, user_sequences, targets, item_codes, num_codebooks=3, max_seq_len=50):
        self.item_codes = item_codes
        self.samples = []
        for uid in user_sequences:
            if uid in targets:
                seq = user_sequences[uid][-max_seq_len:]
                self.samples.append((seq, targets[uid]))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        history, target_item = self.samples[idx]
        input_codes = np.array([self.item_codes[i] for i in history])
        target_codes = self.item_codes[target_item]
        return {"input_codes": torch.LongTensor(input_codes), "target_item": target_item,
                "target_codes": torch.LongTensor(target_codes), "input_length": len(history)}


def collate_sft(batch):
    max_il = max(item["input_length"] for item in batch)
    max_tl = max(item["target_length"] for item in batch)
    nc = batch[0]["input_codes"].shape[-1]
    B = len(batch)
    input_codes = torch.zeros(B, max_il, nc, dtype=torch.long)
    target_codes = torch.zeros(B, max_tl, nc, dtype=torch.long)
    input_lengths = torch.zeros(B, dtype=torch.long)
    target_lengths = torch.zeros(B, dtype=torch.long)
    for i, item in enumerate(batch):
        il, tl = item["input_length"], item["target_length"]
        input_codes[i, :il] = item["input_codes"]
        target_codes[i, :tl] = item["target_codes"]
        input_lengths[i] = il
        target_lengths[i] = tl
    return {"input_codes": input_codes, "target_codes": target_codes,
            "input_lengths": input_lengths, "target_lengths": target_lengths}


def collate_eval(batch):
    max_il = max(item["input_length"] for item in batch)
    nc = batch[0]["input_codes"].shape[-1]
    B = len(batch)
    input_codes = torch.zeros(B, max_il, nc, dtype=torch.long)
    input_lengths = torch.zeros(B, dtype=torch.long)
    target_items = torch.zeros(B, dtype=torch.long)
    target_codes = torch.zeros(B, nc, dtype=torch.long)
    for i, item in enumerate(batch):
        il = item["input_length"]
        input_codes[i, :il] = item["input_codes"]
        input_lengths[i] = il
        target_items[i] = item["target_item"]
        target_codes[i] = item["target_codes"]
    return {"input_codes": input_codes, "input_lengths": input_lengths,
            "target_items": target_items, "target_codes": target_codes}

In [ ]:
%%writefile src/trainers/__init__.py
# Trainers

## 3. Sanity Check

In [ ]:
import sys
sys.path.insert(0, '/content/JDGenRec')

import torch
from src.modules.token_merger import TokenMerger
from src.modules.metrics import TopKAccumulator
from src.models.rqvae import RQVAE

# Token Merger
merger = TokenMerger(embed_dim=64, num_codes=3)
x = torch.randn(2, 5, 3, 64)
assert merger(x).shape == (2, 5, 64)
print("[OK] TokenMerger OK")

# RQVAE
rqvae = RQVAE(input_dim=64, embedding_dim=128, num_codebooks=3, codebook_size=64, hidden_dims=[128])
recon, indices, cl, rl = rqvae(torch.randn(4, 64))
assert indices.shape == (4, 3)
print("[OK] RQVAE OK")

# Metrics
acc = TopKAccumulator(ks=[5, 10])
acc.update([1, 2, 3, 4, 5], 3)
print(f"[OK] Metrics OK: {acc.compute()}")

print("\nAll sanity checks passed!")

## 4. Train RQVAE (Semantic ID Generation)
Trains RQ-VAE on item embeddings to produce 3-level discrete codes (~5 min on T4).

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.auto import tqdm

from src.models.rqvae import RQVAE
from src.data.amazon import AmazonDataset

# Config
SPLIT = "beauty"
DEVICE = "cuda"
EMBEDDING_DIM = 64
RQVAE_DIM = 256
NUM_CODEBOOKS = 3
CODEBOOK_SIZE = 256
RQVAE_EPOCHS = 100
BATCH_SIZE = 256
LR = 2e-4

# Load dataset
dataset = AmazonDataset(data_dir="./data", split=SPLIT)
print(f"Dataset: {dataset.num_items} items, {dataset.num_users} users")

# Get embeddings
item_embeddings = torch.FloatTensor(dataset.get_item_embeddings(EMBEDDING_DIM)[1:])
model = RQVAE(EMBEDDING_DIM, RQVAE_DIM, NUM_CODEBOOKS, CODEBOOK_SIZE, hidden_dims=[512, 256]).to(DEVICE)

# K-means init
print("K-means initialization...")
model.init_codebook_with_kmeans(item_embeddings[:5000].to(DEVICE))

loader = DataLoader(TensorDataset(item_embeddings), batch_size=BATCH_SIZE, shuffle=True)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=RQVAE_EPOCHS, eta_min=LR*0.01)

# Training with gradient clipping and early stopping
best_loss = float('inf')
patience_counter = 0
patience = 20
best_state = None
for epoch in range(RQVAE_EPOCHS):
    model.train()
    total_loss = 0
    for (batch,) in loader:
        batch = batch.to(DEVICE)
        _, _, commit_loss, recon_loss = model(batch)
        loss = recon_loss + commit_loss
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    avg_loss = total_loss / len(loader)
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}/{RQVAE_EPOCHS} | Loss: {avg_loss:.4f}")
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

# Load best model for code generation
if best_state is not None:
    model.load_state_dict(best_state)
print(f"\nBest loss: {best_loss:.4f}")

In [ ]:
# Generate and save item codes
model.eval()
all_codes = []
with torch.no_grad():
    for i in range(0, len(item_embeddings), BATCH_SIZE):
        batch = item_embeddings[i:i+BATCH_SIZE].to(DEVICE)
        codes = model.get_codes(batch)
        all_codes.append(codes.cpu())

all_codes = torch.cat(all_codes, dim=0).numpy()
item_codes = np.zeros((dataset.num_items + 1, NUM_CODEBOOKS), dtype=np.int64)
item_codes[1:] = all_codes

import os
os.makedirs(f"./checkpoints/rqvae_{SPLIT}", exist_ok=True)
np.save(f"./checkpoints/rqvae_{SPLIT}/item_codes.npy", item_codes)

for c in range(NUM_CODEBOOKS):
    print(f"Codebook {c}: {len(np.unique(all_codes[:, c]))}/{CODEBOOK_SIZE} codes used")
code_tuples = [tuple(row) for row in all_codes]
collision_rate = 1.0 - len(set(code_tuples)) / len(code_tuples)
print(f"Collision rate: {collision_rate:.4f}")
print(f"\n[OK] Item codes saved! Shape: {item_codes.shape}")

## 5. Train GenRec (Page-wise NTP + Token Merger)
Main training using Qwen2.5-0.5B with fp16 on T4.

In [ ]:
import math
from torch.utils.data import DataLoader
from src.models.genrec import GenRec
from src.data.genrec_dataset import GenRecSFTDataset, GenRecEvalDataset, collate_sft, collate_eval
from src.modules.metrics import TopKAccumulator

# Config
MODEL_PATH = "Qwen/Qwen2.5-0.5B"  # ~1GB in fp16, fits T4 easily
PAGE_SIZE = 3
MAX_SEQ_LEN = 20  # Use 50 for full experiment
GENREC_EPOCHS = 10  # Use 50 for full experiment
GENREC_BATCH_SIZE = 8
GENREC_LR = 2e-5
BEAM_WIDTH = 10
EVAL_TOPK = 10

# Load data splits
train_seqs, val_targets, test_targets = dataset.get_splits()
print(f"Train users: {len(train_seqs)}, Val: {len(val_targets)}, Test: {len(test_targets)}")

# Create datasets
train_dataset = GenRecSFTDataset(train_seqs, item_codes, NUM_CODEBOOKS, CODEBOOK_SIZE, MAX_SEQ_LEN, PAGE_SIZE)
val_dataset = GenRecEvalDataset(train_seqs, val_targets, item_codes, NUM_CODEBOOKS, MAX_SEQ_LEN)
print(f"Training samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")

train_loader = DataLoader(train_dataset, batch_size=GENREC_BATCH_SIZE, shuffle=True, collate_fn=collate_sft)
val_loader = DataLoader(val_dataset, batch_size=GENREC_BATCH_SIZE, shuffle=False, collate_fn=collate_eval)

In [ ]:
# Initialize model - use float32 for training stability
print(f"Loading {MODEL_PATH}...")
genrec_model = GenRec(
    pretrained_path=MODEL_PATH,
    num_codebooks=NUM_CODEBOOKS,
    codebook_size=CODEBOOK_SIZE,
    use_token_merger=True,
).float().to(DEVICE)

genrec_model.gradient_checkpointing_enable()
print(f"[OK] Model loaded. Vocab size: {len(genrec_model.tokenizer)}")
print(f"Token Merger: {NUM_CODEBOOKS} x {genrec_model.model.config.hidden_size} -> {genrec_model.model.config.hidden_size}")
print(f"Model dtype: {next(genrec_model.model.parameters()).dtype}")
param_mb = sum(p.numel() * p.element_size() for p in genrec_model.parameters()) / 1024**2
print(f"Model size: {param_mb:.0f} MB")

In [ ]:
# Training loop
optimizer = AdamW(genrec_model.parameters(), lr=GENREC_LR, weight_decay=0.01)
num_steps = len(train_loader) * GENREC_EPOCHS
warmup_steps = int(num_steps * 0.01)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, num_steps - warmup_steps)
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

print(f"Training for {GENREC_EPOCHS} epochs, {num_steps} total steps")

for epoch in range(GENREC_EPOCHS):
    genrec_model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{GENREC_EPOCHS}")
    for batch in pbar:
        optimizer.zero_grad()
        outputs = genrec_model.forward_sft(
            input_codes=batch["input_codes"].to(DEVICE),
            target_codes=batch["target_codes"].to(DEVICE),
            input_lengths=batch["input_lengths"].to(DEVICE),
            target_lengths=batch["target_lengths"].to(DEVICE),
        )
        loss = outputs["loss"]
        loss.backward()
        torch.nn.utils.clip_grad_norm_(genrec_model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")

## 6. Evaluate with Constrained Beam Search

In [ ]:
genrec_model.eval()
accumulator = TopKAccumulator(ks=[5, 10])

# Build reverse mapping: code tuple -> item_id
code_to_item = {}
for item_id in range(1, len(item_codes)):
    ct = tuple(item_codes[item_id])
    if ct not in code_to_item:
        code_to_item[ct] = item_id

num_hallucinations = 0
num_total = 0
MAX_EVAL_BATCHES = 50  # Evaluate on subset for speed

with torch.no_grad():
    for batch_idx, batch in enumerate(tqdm(val_loader, desc="Evaluating")):
        if batch_idx >= MAX_EVAL_BATCHES:
            break
        input_codes_b = batch["input_codes"].to(DEVICE)
        input_lengths_b = batch["input_lengths"].to(DEVICE)
        target_items_b = batch["target_items"].numpy()

        try:
            sem_ids, scores = genrec_model.generate_beam(
                input_codes_b, input_lengths_b, beam_width=BEAM_WIDTH, topk=EVAL_TOPK
            )
        except Exception as e:
            print(f"Error: {e}")
            continue

        B = input_codes_b.shape[0]
        for b in range(B):
            predictions = []
            for k in range(min(EVAL_TOPK, sem_ids.shape[1])):
                ct = tuple(sem_ids[b, k].cpu().numpy())
                item_id = code_to_item.get(ct, -1)
                if item_id == -1:
                    num_hallucinations += 1
                else:
                    predictions.append(item_id)
                num_total += 1
            accumulator.update(predictions, target_items_b[b])

print(f"\n{'='*50}")
print(f"Evaluation Results ({accumulator.count} samples):")
metrics = accumulator.compute()
for name, val in metrics.items():
    print(f"  {name}: {val:.4f}")
if num_total > 0:
    print(f"  Hallucination Rate: {num_hallucinations/num_total:.4f}")
print(f"{'='*50}")

## 7. Save Model

In [ ]:
save_path = f"./checkpoints/genrec_{SPLIT}"
genrec_model.save_pretrained(save_path)
print(f"[OK] Model saved to {save_path}")

# Download from Colab (uncomment to use)
# !zip -r /content/genrec_model.zip {save_path}
# from google.colab import files
# files.download('/content/genrec_model.zip')

---
## Notes
- **T4 uses fp16** (not bf16 — Ampere+ required for bf16)
- **Timing**: RQVAE ~5 min, GenRec 10 epochs ~30 min on T4
- **Full experiment**: Set `GENREC_EPOCHS=50`, `MAX_SEQ_LEN=50`
- **Memory**: Qwen2.5-0.5B in fp16 ≈ 1GB, well within T4's 16GB
- **Paper**: [arXiv:2604.14878](https://arxiv.org/abs/2604.14878) (Zou et al., SIGIR 2026)